In [ ]:
#!git clone https://github.com/Tuna209/SentimentAnalysisWithTransformers.git

In [ ]:
!pip install emoji contractions tiktoken wandb

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/nanoGPT/master/model.py -O /content/model.py

In [ ]:
#!pip uninstall -y google-cloud-aiplatform

In [ ]:
#!ls -la /content/model.py

In [ ]:
import os
print(os.listdir())  # Check if model.py is listed

In [ ]:
# Standard libraries
import os
import re
import math
import string
import datetime
from collections import Counter

# Data manipulation and analysis
import numpy as np
import pandas as pd
import scipy.sparse

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP processing
import nltk
import emoji
import contractions
import tiktoken
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer

# Deep learning frameworks
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# Pre-trained models and components
from transformers import BertTokenizer
from model import GPT, GPTConfig, Block, LayerNorm  #https://github.com/karpathy/nanoGPT/blob/master/model.py



In [ ]:
# Platforms and services
#from google.colab import userdata
#import wandb

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
#from google.colab import files
#uploaded = files.upload()
#print(uploaded)

In [ ]:
CONTENT_FILE = r'/content'
SEED = 123

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
#!apt-get update
#!apt-get install -y cuda-toolkit-12-0

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

Data Loading and Preprocessing for Sentiment Analysis

In [ ]:
#train_df = pd.read_csv(os.path.join(CONTENT_FILE, 'train.csv'))
#test_df = pd.read_csv(os.path.join(CONTENT_FILE, 'test.csv'))
train_df = pd.read_csv('train.csv' )
test_df = pd.read_csv('test.csv')

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=SEED)

BASIC ANALYSIS

In [ ]:
print("Train dataset shape:", train_df.shape)
print("Validation dataset shape:", val_df.shape)
print("Test dataset shape:", test_df.shape)

In [ ]:
#print("\nColumns and data types")
#print(train_df.dtypes)  # All objects no need to rerun
print("\nFirst row:")
print(train_df.head(1))
#print("\nMissing values:")
#print(train_df.isnull().sum()) # No nulls, no need to rerun
print("\nSummary statistics:")
print(train_df.describe())

Sentiment Distribution

In [ ]:
plt.style.use('ggplot')
sns.set(font_scale=1.2)

# Distribution of sentiment classes
plt.figure(figsize=(12, 6))
sentiment_counts = train_df['customer_sentiment'].value_counts()
sentiment_counts.plot(kind='bar', color=sns.color_palette("viridis"))
plt.title('Distribution of Customer Sentiment Classes')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('sentiment_distribution.png')
plt.show()

# Percentage distribution
print("Percentage distribution of sentiment classes:")
print(train_df['customer_sentiment'].value_counts(normalize=True) * 100)

# Compare sentiment distribution across datasets
plt.figure(figsize=(12, 8))
train_pct = train_df['customer_sentiment'].value_counts(normalize=True) * 100
val_pct = val_df['customer_sentiment'].value_counts(normalize=True) * 100
test_pct = test_df['customer_sentiment'].value_counts(normalize=True) * 100

# Create a DataFrame for easy plotting
sentiment_comparison = pd.DataFrame({
    'Train': train_pct,
    'Validation': val_pct,
    'Test': test_pct
})

sentiment_comparison.plot(kind='bar', figsize=(12, 6))
plt.title('Sentiment Distribution Comparison Across Datasets')
plt.xlabel('Sentiment')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('sentiment_comparison.png')
plt.show()

# Calculate class weights for handling imbalance (if needed)
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_df['customer_sentiment']),
    y=train_df['customer_sentiment']
)
class_weight_dict = dict(zip(np.unique(train_df['customer_sentiment']), class_weights))
print("Class weights for handling imbalance:")
print(class_weight_dict)

Here we see the call center is generally called for complaints, but not positive remarks.

In [ ]:
train_df['data_split_type'] = "train"
val_df['data_split_type'] = "val"
test_df['data_split_type'] = "test"
df = pd.concat([train_df,val_df, test_df])
df.reset_index(drop=True, inplace=True)
df['sentiment'] = df['customer_sentiment'].map({'positive': 0, 'neutral': 1, 'negative': 2})
df['text_cleaned'] = df.conversation

In [ ]:
#print("sentiment unique values:")
#print(df['text_cleaned'].unique())
print("\nValue counts:")
print(df['text_cleaned'].value_counts())

#print("text_cleaned unique values:")
#print(df['text_cleaned'].unique())
print("\nValue counts:")
print(df['text_cleaned'].value_counts())

In [ ]:
df.columns

In [ ]:
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_colwidth', None)  # Show full width of text in cells

df['text_cleaned'].head(1)

In [ ]:
def clean_initial_agent_greeting(text):
    lines = text.split('\n')

    # Check if the first line starts with "Agent:"
    if lines and lines[0].strip().startswith("Agent:"):
        # Remove the first line
        lines = lines[1:]

    # Join the remaining lines back together
    cleaned_text = '\n'.join(lines)

    return cleaned_text

df['text_cleaned'] = df['conversation'].apply(clean_initial_agent_greeting)

In [ ]:
def clean_all_agent_lines(text):
    lines = text.split('\n')

    # Keep only lines that do not start with "Agent:"
    cleaned_lines = [line for line in lines if not line.strip().startswith("Agent:")]

    # Join the filtered lines back together
    cleaned_text = '\n'.join(cleaned_lines)

    return cleaned_text

df['text_cleaned'] = df['conversation'].apply(clean_all_agent_lines)

In [ ]:
df['text_cleaned'].head()

In [ ]:
df['text_cleaned'].head()

In [ ]:
df['text_cleaned'] = df['text_cleaned'].str.lower()

In [ ]:
# Delete agent/customer script
df['text_cleaned'] = df['text_cleaned'].str.replace(r"\b(Agent:|Customer:)\s*", "", regex=True)

# Remove redundant web info
df['text_cleaned'] = df['text_cleaned'].apply(lambda text: re.sub(r"\S+@\S+|www\.\S+\.com", "", text))

# Remove punctuation, numbers, extra spaces and replacing repetitions of punctuation
df['text_cleaned'] = df['text_cleaned'].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation)))
df['text_cleaned'] = df['text_cleaned'].apply(lambda x: re.sub(r'\d+', '', x))
df['text_cleaned'] = df['text_cleaned'].apply(lambda x: ' '.join(x.split()))
df['text_cleaned'] = df['text_cleaned'].apply(lambda x: re.sub(r'(\W)\1+', r'\1', x))


In [ ]:
# Removing stop words:
stop_words_set = set(stopwords.words('english'))
no_stopwords = []
for sentence in df["text_cleaned"]:
    # Use the new variable name here
    no_stopwords.append(' '.join(word for word in sentence.split() if word not in stop_words_set))
df["text_cleaned"]=no_stopwords

In [ ]:
# Convert Emojis to Words
def convert_emojis_to_words(text):
    converted_text = emoji.demojize(text)
    return converted_text
df['text_cleaned'] = df['text_cleaned'].apply(convert_emojis_to_words)

# Removing special characters
df['text_cleaned'] = df['text_cleaned'].apply(lambda x: re.sub(r"[^\w\s]", '', x))

# Removing contractions
df['text_cleaned'] = df['text_cleaned'].apply(lambda x: contractions.fix(x))


In [ ]:
# Lemmatization
lemmatizer = WordNetLemmatizer()
def lemmatize_words(text):
    return " ".join([lemmatizer.lemmatize(word) for word in text.split()])

df["text_cleaned"] = df["text_cleaned"].apply(lambda text: lemmatize_words(text))

Stemming after Lemmatization might be redundant, I left this part for the latest trials.

In [ ]:
# Initialize the stemmer
#stemmer = PorterStemmer()

#def stem_words(text):
    return " ".join([stemmer.stem(word) for word in text.split()])

# Apply stemming to the already lemmatized text
#df["text_cleaned"] = df["text_cleaned"].apply(lambda text: stem_words(text))

In [ ]:
df.columns

In [ ]:
# Compute word count for each conversation
df['word_count'] = df['text_cleaned'].apply(lambda x: len(x.split()))

# Filter out short conversations from training set
df = df[~((df['word_count'] < 8) & (df['data_split_type'] == 'train'))]

# Create word count distribution visualization with different styling
plt.figure(figsize=(12, 7))
sns.histplot(data=df, x='word_count', bins=50, color='teal', kde=True)
plt.title('Word Count Distribution in Conversations', fontsize=14)
plt.xlabel('Word Count', fontsize=12)
plt.ylabel('Number of Conversations', fontsize=12)
plt.axvline(x=df['word_count'].median(), color='crimson', linestyle='--',
            label=f'Median: {df["word_count"].median():.0f} words')
plt.axvline(x=df['word_count'].mean(), color='darkgreen', linestyle='-.',
            label=f'Mean: {df["word_count"].mean():.0f} words')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Check sentiment distribution
sentiment_counts = df['customer_sentiment'].value_counts()
print("Sentiment distribution:")
print(sentiment_counts)
print("\nPercentage distribution:")
print(sentiment_counts / len(df) * 100)

# Visualize the distribution
plt.figure(figsize=(10, 6))
sentiment_counts.plot(kind='bar', color=sns.color_palette("viridis"))
plt.title('Distribution of Customer Sentiment')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate word count for each conversation
df['word_count'] = df['conversation'].apply(lambda x: len(x.split()))

# Word count statistics by sentiment
word_count_by_sentiment = df.groupby('customer_sentiment')['word_count'].agg(['mean', 'median', 'min', 'max'])
print("Word count statistics by sentiment:")
print(word_count_by_sentiment)

# Plot word count distribution by sentiment
plt.figure(figsize=(12, 6))
sns.boxplot(x='customer_sentiment', y='word_count', data=df, palette='viridis')
plt.title('Word Count Distribution by Sentiment')
plt.xlabel('Customer Sentiment')
plt.ylabel('Word Count')
plt.grid(axis='y', alpha=0.3)
plt.show()


In [ ]:
# Display split sizes
print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)}")

# Check sentiment distribution in each split
print("\nSentiment distribution in training set:")
print(train_df['customer_sentiment'].value_counts())
print("\nSentiment distribution in validation set:")
print(val_df['customer_sentiment'].value_counts())
print("\nSentiment distribution in test set:")
print(test_df['customer_sentiment'].value_counts())

In [ ]:
!pip install dotenv

In [ ]:
import wandb
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

def init_wandb():
    # Get API key from environment variable
    api_key = os.getenv('WANDB_API_KEY')

    if not api_key:
        print("Please set your WANDB_API_KEY environment variable")
        print("You can get your API key from https://wandb.ai/settings")
        return False

    # Initialize wandb
    try:
        wandb.login(key=api_key)
        print("Successfully logged in to Weights & Biases")
        return True
    except Exception as e:
        print(f"Error logging in to Weights & Biases: {e}")
        return False

project_name = "customer_service_sentiment_analysis"

# Model Design

In [ ]:
# 1. Full Self-Attention for non-causal processing
class FullSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # non-causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=False)
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

# 2. SentimentBlock using FullSelfAttention
class SentimentBlock(Block):
    def __init__(self, config):
        super().__init__(config)
        self.attn = FullSelfAttention(config)

# 3. SentimentTransformer model for sentiment classification
class SentimentTransformer(GPT):
    def __init__(self, config, num_classes=3): # The output of the sentiment analysis will be 0, 1, 2.
        super().__init__(config)

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([SentimentBlock(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        # Override the lm_head with a new linear layer for sentiment classification
        self.lm_head = torch.nn.Linear(config.n_embd, num_classes, bias=False)

    def forward(self, idx):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device)  # shape (t)

        # Forward the GPT model
        tok_emb = self.transformer.wte(idx)  # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos)  # position embeddings of shape (t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)

        # Aggregate sequence representations (average over the sequence)
        x = torch.mean(x, dim=1)

        # Pass through the modified linear layer to get class logits
        logits = self.lm_head(x)

        return logits

    @classmethod
    def from_pretrained(cls, model_type, override_args=None):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        override_args = override_args or {} # default to empty dict
        # only dropout can be overridden see more notes below
        assert all(k == 'dropout' for k in override_args)
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        print("forcing vocab_size=50257, block_size=1024, bias=True")
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        config_args['bias'] = True # always True for GPT model checkpoints
        # we can override the dropout rate, if desired
        if 'dropout' in override_args:
            print(f"overriding dropout rate to {override_args['dropout']}")
            config_args['dropout'] = override_args['dropout']
        # create a from-scratch initialized model
        config = GPTConfig(**config_args)
        model = SentimentTransformer(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                if k != 'lm_head.weight': # Except Last layer
                    assert sd_hf[k].shape == sd[k].shape
                    with torch.no_grad():
                        sd[k].copy_(sd_hf[k])

        return model

# Define Dataset Classes

In [ ]:
# Dataset class for GPT model
class CustomerSentimentDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256, text_column='text_cleaned', sentiment_column='sentiment'):
        self.data = df
        self.texts = self.data[text_column].values
        self.labels = self.data[sentiment_column].values
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        # Use the tokenizer's method to encode the text
        input_ids = self.tokenizer.encode(text)

        # Ensure the sequence is at most max_length
        input_ids = input_ids[:self.max_length]

        # Padding if necessary to ensure all sequences are of the same length
        padding_length = self.max_length - len(input_ids)
        if padding_length > 0:
            # Append zeros at the end for padding
            input_ids = input_ids + [0] * padding_length

        # Ensure it returns a torch tensor
        input_ids = torch.tensor(input_ids, dtype=torch.long)

        return input_ids, torch.tensor(label, dtype=torch.long)

In [ ]:
# Assigning tokinezer
tokenizer = tiktoken.get_encoding("gpt2")

# Load the dataset
train_dataset = CustomerSentimentDataset(df[df.data_split_type == 'train'], max_length=1024, tokenizer=tokenizer)
test_dataset = CustomerSentimentDataset(df[df.data_split_type == 'test'], max_length=1024, tokenizer=tokenizer)
val_dataset = CustomerSentimentDataset(df[df.data_split_type == 'val'], max_length=1024, tokenizer=tokenizer)

# Define Evaluation and Visualization Functions

In [ ]:
def evaluate(model, loader):
    """Evaluate model performance on a data loader"""
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            logits = model(inputs)
            loss = F.cross_entropy(logits, labels)

            # Gather statistics
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate metrics
    avg_loss = total_loss / len(loader)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    precision_macro = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall_macro = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    conf_matrix = confusion_matrix(all_labels, all_preds)
    accuracy = accuracy_score(all_labels, all_preds)

    return avg_loss, f1_macro, precision_macro, recall_macro, accuracy, conf_matrix

def plot_confusion_matrix(cm, class_names):
    """Create a confusion matrix plot"""
    figure = plt.figure(figsize=(8, 8))
    sns.heatmap(cm, annot=True, fmt="d", cmap=plt.cm.Blues, cbar=False,
                xticklabels=class_names, yticklabels=class_names)
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.close()
    return figure

# Define Training and Evaluation Function

In [ ]:
def train_and_evaluate():
    """Train and evaluate the model"""
    # Initialize wandb with timestamp in run name
    current_time = datetime.datetime.now().strftime("%d-%m-%Y_%H-%M-%S")
    wandb.init(project="customer_service_sentiment_analysis", name=f"run_{current_time}")
    config = wandb.config

    # Set device
    if torch.cuda.is_available():
        device = torch.device('cuda')
    else:
        device = torch.device('cpu')
    print(f"Using device: {device}")

    # Create model based on configuration
    if config.init_from == 'scratch':
        model_config = GPTConfig(vocab_size=config.vocab_size, block_size=config.block_size,
                            n_layer=config.n_layer, n_head=config.n_head,
                            n_embd=config.n_embd, dropout=config.dropout)
        model = SentimentTransformer(model_config).to(device)
    elif str(config.init_from).startswith('gpt'):
        model_config = dict(dropout=config.dropout)
        model = SentimentTransformer.from_pretrained(model_type=config.init_from, override_args=model_config).to(device)
    else:
        print('Please select correct model!')
        return

    # Configure optimizer
    optimizer = model.configure_optimizers(weight_decay=config.weight_decay,
                                         learning_rate=config.learning_rate,
                                         betas=(config.beta1, config.beta2),
                                         device_type=device)

    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

    # Tracking variables
    best_f1_macro = 0
    best_val_loss = float('inf')
    best_model_path = f"best_model_{current_time}.pt"

    # Training loop
    for epoch in range(config.epochs):
        model.train()
        # Initialize lists to store batch metrics
        train_losses, train_f1s, train_precs, train_recs, train_accs = [], [], [], [], []

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass and loss calculation
            optimizer.zero_grad()
            logits = model(inputs)
            loss = F.cross_entropy(logits, labels)

            # Backward pass and optimization
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.max_grad_norm)
            optimizer.step()

            # Calculate and store batch metrics
            train_losses.append(loss.item())
            predictions = logits.argmax(dim=1).cpu().numpy()
            labels_np = labels.cpu().numpy()
            train_f1s.append(f1_score(labels_np, predictions, average='macro', zero_division=0))
            train_precs.append(precision_score(labels_np, predictions, average='macro', zero_division=0))
            train_recs.append(recall_score(labels_np, predictions, average='macro', zero_division=0))
            train_accs.append(accuracy_score(labels_np, predictions))

        # Calculate average metrics over all batches
        avg_train_loss = np.mean(train_losses)
        avg_train_f1 = np.mean(train_f1s)
        avg_train_prec = np.mean(train_precs)
        avg_train_rec = np.mean(train_recs)
        avg_train_acc = np.mean(train_accs)

        # Evaluation phase
        model.eval()
        val_loss, val_f1, val_prec, val_rec, val_acc, val_conf_matrix = evaluate(model, val_loader)

        # Print progress
        print(f"\nEpoch {epoch+1}, Train Loss: {avg_train_loss:.2f}, Train F1 Macro: {avg_train_f1:.2f}, "
              f"Train Precision Macro: {avg_train_prec:.2f}, Train Recall Macro: {avg_train_rec:.2f}, "
              f"Train Accuracy: {avg_train_acc:.2f}")
        print(f"Epoch {epoch+1}, Val Loss: {val_loss:.2f}, Val F1 Macro: {val_f1:.2f}, "
              f"Val Precision Macro: {val_prec:.2f}, Val Recall Macro: {val_rec:.2f}, "
              f"Val Accuracy: {val_acc:.2f}")

        # Saving the best F1 macro result model
        if val_f1 > best_f1_macro:
            best_f1_macro = val_f1
            checkpoint = {
                'model': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'model_args': model_config,
                'epoch_num': epoch+1,
                'best_val_loss': best_val_loss,
                'config': {k: v for k, v in dict(wandb.config).items()},
            }
            torch.save(checkpoint, best_model_path)
            print(f"New best model saved with validation F1: {val_f1:.4f}")

        # Log metrics to wandb
        wandb.log({
            "epoch": epoch + 1,
            "train_loss": avg_train_loss,
            "train_f1_macro": avg_train_f1,
            "train_precision_macro": avg_train_prec,
            "train_recall_macro": avg_train_rec,
            "train_accuracy": avg_train_acc,
            "val_loss": val_loss,
            "val_f1_macro": val_f1,
            "val_precision_macro": val_prec,
            "val_recall_macro": val_rec,
            "val_accuracy": val_acc
        })

    # Load the best model and evaluate on test data
    checkpoint = torch.load(best_model_path, weights_only=False)
    model.load_state_dict(checkpoint['model'])
    test_loss, test_f1, test_prec, test_rec, test_acc, test_conf_matrix = evaluate(model, test_loader)
    print(f"\nBest model epoch {checkpoint['epoch_num']} and test results, "
          f"Test Loss: {test_loss:.2f}, Test F1 Macro: {test_f1:.2f}, "
          f"Test Precision Macro: {test_prec:.2f}, Test Recall Macro: {test_rec:.2f}, "
          f"Test Accuracy: {test_acc:.2f}")

    # Plot and log confusion matrix
    fig = plot_confusion_matrix(test_conf_matrix, class_names=['Positive', 'Neutral', 'Negative'])
    wandb.log({"test_confusion_matrix": wandb.Image(fig)})

    # Log test results to wandb
    wandb.log({
        "test_loss": test_loss,
        "test_f1_macro": test_f1,
        "test_precision_macro": test_prec,
        "test_recall_macro": test_rec,
        "test_accuracy": test_acc
    })

    wandb.finish()

# Setup for Phase 1 - Training from Scratch

In [ ]:


# Configure the sweep for training from scratch
sweep_config = {
    'method': 'grid',
    'parameters': {
        'epochs': {
            'value': 30
        },
        'batch_size': {
            'values': [32]
        },
        'vocab_size': {
            'values': [50304]
        },
        'block_size': {
            'values': [1024]
        },
        'n_layer': {
            'values': [6]
        },
        'n_head': {
            'values': [6]
        },
        'n_embd': {
            'values': [384]
        },
        'dropout': {
           'values': [0.3]
        },
        'learning_rate': {
            'values': [0.0001]
        },
        'weight_decay': {
            'values': [0.01]
        },
        'max_grad_norm': {
            'values': [1.0]
        },
        'beta1': {
            'values': [0.9]
        },
        'beta2': {
            'values': [0.999]
        },
        'init_from': {
            'value': 'scratch'
        },
    }
}

# Initialize the sweep and start the agent
sweep_id = wandb.sweep(sweep_config, project=project_name)
wandb.agent(sweep_id=sweep_id, function=train_and_evaluate)

# Setup for Phase 2 - Fine-tuning GPT-2

In [ ]:
# Configure the sweep for fine-tuning GPT-2
sweep_config = {
    'method': 'grid',
    'parameters': {
        'epochs': {
            'value': 30
        },
        'batch_size': {
            'values': [32]
        },
        'dropout': {
           'values': [0.3]
        },
        'learning_rate': {
            'values': [0.001, 0.0005]
        },
        'weight_decay': {
            'values': [0.1]
        },
        'max_grad_norm': {
            'values': [1.0]
        },
        'beta1': {
            'values': [0.9]
        },
        'beta2': {
            'values': [0.999]
        },
        'init_from': {
            'value': 'gpt2'
        },
    }
}

# Initialize the sweep and start the agent
sweep_id = wandb.sweep(sweep_config, project=project_name)
wandb.agent(sweep_id=sweep_id, function=train_and_evaluate)